# PathSnap - P2: Model Experimentation + Bayesian Optimisation
Input: P1's `candidates_{train,student}.parquet` (`trip_id, obs_idx, arc_id, label` + features).

**Method:** 5 models, 3-fold GroupKFold (by trip) on a fixed-seed random 30-trip subset of the 80 training trips, student-test = 10 trips, Optuna TPE on LightGBM (15 min timeout), final model refit on all 80 trips. Evaluator trips are never loaded.

In [ ]:
!pip -q install optuna lightgbm pyarrow

In [ ]:
import time, json, os, zipfile, warnings
import numpy as np, pandas as pd, joblib, matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (f1_score, log_loss, accuracy_score, precision_score,
                             recall_score, ConfusionMatrixDisplay)
from lightgbm import LGBMClassifier
import optuna
warnings.filterwarnings('ignore')

SEED = 42
TRAIN_PATH   = 'candidates_train.parquet'
STUDENT_PATH = 'candidates_student.parquet'
META = ['trip_id', 'obs_idx', 'arc_id', 'label']
DROP = ['from_node', 'to_node', 'snap_x', 'snap_y']   # per-trip ids / absolute coordinates
N_FOLDS = 3
N_SUBSET_TRIPS = 30        # CV + tuning subset of the 80 train trips
N_TRIALS = 20
OPT_TIMEOUT = 900          # seconds
BEST = 'LightGBM (boosting)'
os.makedirs('results', exist_ok=True); os.makedirs('models', exist_ok=True)

In [ ]:
# Colab: upload both parquet files if they are not on disk yet (wait for the upload to finish)
if not (os.path.exists(TRAIN_PATH) and os.path.exists(STUDENT_PATH)):
    from google.colab import files
    files.upload()

In [ ]:
def check_parquet(p):
    assert os.path.exists(p), f'{p} not found - upload it first'
    with open(p, 'rb') as f:
        head = f.read(4); f.seek(-4, 2); tail = f.read(4)
    assert head == b'PAR1' and tail == b'PAR1', f'{p} is truncated/corrupt ({os.path.getsize(p)} bytes) - re-upload'
check_parquet(TRAIN_PATH); check_parquet(STUDENT_PATH)

tr = pd.read_parquet(TRAIN_PATH).reset_index(drop=True)
st = pd.read_parquet(STUDENT_PATH).reset_index(drop=True)
# raw features only: ids, absolute coords and the pre-scaled duplicates (*_scaled) are excluded
FEATURES = [c for c in tr.columns if c not in META + DROP and not c.endswith('_scaled')
            and pd.api.types.is_numeric_dtype(tr[c])]
print(len(FEATURES), 'features:', FEATURES)
print('train', tr.shape, tr.trip_id.nunique(), 'trips | student', st.shape, st.trip_id.nunique(), 'trips')
assert set(tr.trip_id).isdisjoint(set(st.trip_id)), 'trip leakage between partitions!'
def miss_rate(d): return 1 - d.groupby(['trip_id', 'obs_idx']).label.max().mean()
print('pos rate %.4f | candidate miss rate  train %.4f | student %.4f' % (tr.label.mean(), miss_rate(tr), miss_rate(st)))

In [ ]:
# float32 + fixed-seed random subset of trips for model comparison and tuning
if 'tr_full' not in globals():
    for d in (tr, st):
        f = d.select_dtypes('float64').columns; d[f] = d[f].astype('float32')
    rng = np.random.default_rng(SEED)
    keep = rng.choice(np.sort(tr.trip_id.unique()), N_SUBSET_TRIPS, replace=False)
    tr_full = tr                                              # all 80 trips: final refit only
    tr = tr[tr.trip_id.isin(keep)].reset_index(drop=True)     # subset: CV + comparison + Optuna
print('subset', tr.shape, tr.trip_id.nunique(), 'trips | full', tr_full.shape, tr_full.trip_id.nunique(), 'trips')

## Metrics
Top-1 is observation-level (best-scoring candidate per GPS point; reference arc missing from candidates = wrong), over points that have at least one candidate. Precision/recall/F1/log loss are candidate-level at threshold 0.5.

In [ ]:
def pick_top1(df, p):
    d = df[['trip_id', 'obs_idx', 'arc_id', 'label']].copy(); d['p'] = p
    return d.loc[d.groupby(['trip_id', 'obs_idx'])['p'].idxmax()]

def build_adj(arcs):   # arcs: (n,2) [from_node,to_node]; arc_id = row index
    from collections import defaultdict
    by_from = defaultdict(list)
    for i, (a, b) in enumerate(arcs): by_from[a].append(i)
    return {i: set(by_from[b]) for i, (a, b) in enumerate(arcs)}

ADJ = None   # dict trip_id -> build_adj(arcs); None => route continuity reported as N/A

def continuity(picks, adj=None):
    adj = ADJ if adj is None else adj
    if adj is None: return np.nan
    ok = tot = 0
    for t, g in picks.sort_values(['trip_id', 'obs_idx']).groupby('trip_id'):
        a, nxt = g.arc_id.values, adj[t]
        for x, y in zip(a[:-1], a[1:]):
            tot += 1; ok += (x == y) or (y in nxt.get(x, ()))
    return ok / tot if tot else np.nan

def evaluate(model, df):
    X, y = df[FEATURES].values, df.label.values
    t = time.perf_counter(); p = model.predict_proba(X)[:, 1]; inf = time.perf_counter() - t
    pred = p >= .5; picks = pick_top1(df, p)
    return dict(top1=picks.label.sum() / df.groupby(['trip_id', 'obs_idx']).ngroups,
                accuracy=accuracy_score(y, pred),
                precision_pos=precision_score(y, pred, zero_division=0), recall_pos=recall_score(y, pred, zero_division=0),
                precision_macro=precision_score(y, pred, average='macro', zero_division=0),
                recall_macro=recall_score(y, pred, average='macro', zero_division=0),
                macro_f1=f1_score(y, pred, average='macro'),
                logloss=log_loss(y, np.clip(p, 1e-7, 1 - 1e-7), labels=[0, 1]),
                continuity=continuity(picks), inf_ms_per_1k=inf / len(df) * 1e6), p

## 5 models, 5 families

In [ ]:
MODELS = {
 'LogReg (linear)': make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)),
 'DecisionTree (tree)': DecisionTreeClassifier(max_depth=12, min_samples_leaf=50, random_state=SEED),
 'RandomForest (bagging)': RandomForestClassifier(n_estimators=100, max_depth=16, min_samples_leaf=2, n_jobs=-1, random_state=SEED),
 'LightGBM (boosting)': LGBMClassifier(n_estimators=300, learning_rate=0.1, num_leaves=31, random_state=SEED, verbose=-1),
 'MLP (neural)': make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-4, early_stopping=True, max_iter=50, random_state=SEED)),
}
KEY_CFG = {
 'LogReg (linear)': 'C=1, lbfgs, scaled',
 'DecisionTree (tree)': 'max_depth=12, min_samples_leaf=50, gini',
 'RandomForest (bagging)': '100 trees, max_depth=16, min_samples_leaf=2',
 'LightGBM (boosting)': '300 est, lr=0.1, num_leaves=31',
 'MLP (neural)': '(64,32), adam, early stop, max_iter=50, scaled'}

In [ ]:
gkf = GroupKFold(n_splits=N_FOLDS)
Xtr, ytr, gtr = tr[FEATURES].values, tr.label.values, tr.trip_id.values
rows, fitted, student_p = [], {}, {}
for name, m in MODELS.items():
    cvf1, cvt1 = [], []
    for a, b in gkf.split(Xtr, ytr, gtr):
        r, _ = evaluate(clone(m).fit(Xtr[a], ytr[a]), tr.iloc[b]); cvf1.append(r['macro_f1']); cvt1.append(r['top1'])
    t0 = time.perf_counter(); full = clone(m).fit(Xtr, ytr); fit_s = time.perf_counter() - t0
    rtr, _ = evaluate(full, tr); rst, p = evaluate(full, st)
    fitted[name], student_p[name] = full, p
    rows.append(dict(model=name, config=KEY_CFG[name], cv_f1_mean=np.mean(cvf1), cv_f1_std=np.std(cvf1), cv_top1=np.mean(cvt1),
                     train_top1=rtr['top1'], train_s=fit_s,
                     **{'st_' + k: v for k, v in rst.items()}))
    print(name, {k: round(float(v), 4) for k, v in rows[-1].items() if isinstance(v, (int, float, np.floating))})
    joblib.dump(full, f"models/{name.split()[0]}.joblib")
comp = pd.DataFrame(rows).round(4); comp.to_csv('results/model_comparison.csv', index=False); comp

In [ ]:
n = comp.model.str.split().str[0]; x = np.arange(len(n)); w = .38
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].bar(n, comp.cv_f1_mean, yerr=comp.cv_f1_std); ax[0].set_title(f'CV Macro F1 (mean±std, {N_FOLDS}-fold group, {N_SUBSET_TRIPS} trips)')
ax[1].bar(x - w/2, comp.st_top1, w, label='Top-1'); ax[1].bar(x + w/2, comp.st_macro_f1, w, label='Macro F1')
ax[1].set_xticks(x); ax[1].set_xticklabels(n); ax[1].legend(); ax[1].set_title('Student-test')
ax[2].bar(n, comp.st_logloss); ax[2].set_title('Student-test Log Loss')
for a in ax: a.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.savefig('results/model_comparison.png', dpi=150); plt.show()

fig, ax = plt.subplots(1, len(student_p), figsize=(4 * len(student_p), 3.6))
for a, (k, p) in zip(ax, student_p.items()):
    ConfusionMatrixDisplay.from_predictions(st.label, p >= .5, ax=a, colorbar=False); a.set_title(k.split()[0])
plt.tight_layout(); plt.savefig('results/confusion_matrices.png', dpi=150); plt.show()

mlp = fitted['MLP (neural)'][-1]      # only model with an epoch-wise history
plt.plot(mlp.loss_curve_, label='train loss')
if getattr(mlp, 'validation_scores_', None) is not None: plt.plot(mlp.validation_scores_, label='val score (accuracy)')
plt.legend(); plt.xlabel('epoch'); plt.title('MLP training history'); plt.savefig('results/mlp_curve.png', dpi=150); plt.show()

## Bayesian optimisation (Optuna TPE, group CV on the 30-trip subset)
`BEST` is the model chosen for tuning (cheap to tune, tied with the top models on CV). Change it and the search space if you pick another model. Objective: mean CV macro F1.

In [ ]:
def make_lgbm(**kw): return LGBMClassifier(random_state=SEED, verbose=-1, subsample_freq=1, **kw)

def objective(trial):
    params = dict(
        n_estimators=trial.suggest_int('n_estimators', 100, 600),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        num_leaves=trial.suggest_int('num_leaves', 15, 127),
        max_depth=trial.suggest_int('max_depth', 3, 12),
        min_child_samples=trial.suggest_int('min_child_samples', 5, 100),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 10, log=True))
    f1s = []
    for a, b in gkf.split(Xtr, ytr, gtr):
        r, _ = evaluate(make_lgbm(**params).fit(Xtr[a], ytr[a]), tr.iloc[b]); f1s.append(r['macro_f1'])
    return float(np.mean(f1s))

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
t0 = time.perf_counter(); study.optimize(objective, n_trials=N_TRIALS, timeout=OPT_TIMEOUT); opt_s = time.perf_counter() - t0
study.trials_dataframe().to_csv('results/optuna_trials.csv', index=False)
json.dump(study.best_params, open('results/best_params.json', 'w'), indent=2)
n_done = sum(t.state.name == 'COMPLETE' for t in study.trials)
json.dump(dict(planned_trials=N_TRIALS, completed_trials=n_done, timeout_s=OPT_TIMEOUT, elapsed_s=round(opt_s, 1),
               best_cv_macro_f1=study.best_value, objective='mean CV macro F1', folds=N_FOLDS, subset_trips=N_SUBSET_TRIPS),
          open('results/optuna_summary.json', 'w'), indent=2)
print(f'{n_done}/{N_TRIALS} trials completed in {opt_s:.0f}s | best CV macro F1 {study.best_value:.4f}'); print(study.best_params)

In [ ]:
# tuned vs untuned: both refit on ALL train trips, same partitions and metrics, evaluated on the student-test trips
Xfull, yfull = tr_full[FEATURES].values, tr_full.label.values
untuned = clone(MODELS[BEST]).fit(Xfull, yfull)
tuned = make_lgbm(**study.best_params).fit(Xfull, yfull)
def cv_f1(m):   # CV macro F1 on the same 30-trip folds (optimistic for tuned: Optuna chose params on these folds)
    return np.mean([evaluate(clone(m).fit(Xtr[a], ytr[a]), tr.iloc[b])[0]['macro_f1'] for a, b in gkf.split(Xtr, ytr, gtr)])
out = []
for nm, m in [('untuned', untuned), ('tuned', tuned)]:
    r, _ = evaluate(m, st); out.append(dict(model=nm, cv_macro_f1=cv_f1(m), **r))
tt = pd.DataFrame(out).round(4); tt.to_csv('results/tuned_vs_untuned.csv', index=False); tt

In [ ]:
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances
plot_optimization_history(study); plt.savefig('results/optuna_history.png', dpi=150, bbox_inches='tight'); plt.show()
plot_param_importances(study);    plt.savefig('results/optuna_importances.png', dpi=150, bbox_inches='tight'); plt.show()

# handoff artifacts for P3 / P4: tuned model refit on all 80 trips + exact raw feature order
joblib.dump(tuned, 'models/final_tuned_raw.joblib')
json.dump(FEATURES, open('models/features_raw.json', 'w'))
json.dump(dict(seed=SEED, folds=N_FOLDS, subset_trips=[str(t) for t in sorted(tr.trip_id.unique())],
               n_train_trips_final_fit=int(tr_full.trip_id.nunique()), features=FEATURES, best=BEST),
          open('results/run_config.json', 'w'), indent=2)
print('final model features:', FEATURES)

In [ ]:
with zipfile.ZipFile('p2_outputs.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk('results'):
        for f in fs: z.write(os.path.join(root, f))
    z.write('models/final_tuned_raw.joblib'); z.write('models/features_raw.json')
try:
    from google.colab import files; files.download('p2_outputs.zip')
except ImportError:
    pass